# RG3 적은 모델 결합과 RF 오탐 확인
3개 후보의 사전고정 결합, 모델 후보선택에 반복평가 영향이 있음.

In [1]:
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
setup=''.join(json.loads((root/'notebooks/27_rg3_modern_ensembles.ipynb').read_text(encoding='utf-8'))['cells'][1]['source']).split('exec(src)')[0]
exec(setup)
exec(src.split('scores=[];logs=[];errors=[]')[0])
members=['RandomForest','ExtraTrees','CatBoost']
records=[]
def cutoff(s,yy,rate):
    normal=s[(yy==0)&np.isfinite(s)]
    return float(np.quantile(normal,1-rate,method='higher')) if len(normal) else 1.
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=np.column_stack([model_pair(name,ref,q).max(axis=1) for name in members])
        inner=np.full((len(ref),3),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                for k,name in enumerate(members):inner[mask,k]=model_pair(name,ref[~mask],ref[mask]).max(axis=1)
        blend_specs={'RF_ET_mean':np.array([.5,.5,0]),'RF_Cat_mean':np.array([.5,0,.5]),'RF_ET_Cat_mean':np.ones(3)/3,'RF_half_ET_Cat':np.array([.5,.25,.25]),'RF_baseline':np.array([1,0,0])}
        for name,weight in blend_specs.items():
            score=outer@weight;cal=inner@weight
            cut=cutoff(cal,y[ref],.01)
            pred=score>cut
            for k,z in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),prediction=int(pred[k]),score=float(score[k]),ranking_AP_available=True,RF_cut=cut,confirm_cut=np.nan))
        rf_cut=cutoff(inner[:,0],y[ref],.01)
        for confirmer,k in [('ET',1),('Cat',2)]:
            for rate in [.1,.25]:
                confirm_cut=cutoff(inner[:,k],y[ref],rate)
                pred=(outer[:,0]>rf_cut)&(outer[:,k]>confirm_cut)
                for i,z in enumerate(q):records.append(dict(scheme=scheme,model=f'RF_confirm_{confirmer}_{int(rate*100)}',TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),prediction=int(pred[i]),score=np.nan,ranking_AP_available=False,RF_cut=rf_cut,confirm_cut=confirm_cut))
        print('Completed confirmation',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,AP=average_precision_score(g.actual,g.score) if g.ranking_AP_available.all() else np.nan,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_small_confirm_ensemble_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_small_confirm_ensemble_predictions.csv',index=False,encoding='utf-8-sig')
lines=['# RG3 소규모 앙상블과 오탐 확인 필터','',
'RF/ET/CatBoost 각각기존흔들림4개+관계잔차4개두모델max로정의한다. 앞선15모델및신규모델결과를보고3개를선택했으므로후보선택자체가동일평가자료의영향을받는다. 독립최종평가가아니다.',
'평균결합은고정RF-ET/RF-Cat/3개동일/가중치0.5:0.25:0.25를사전에정해비교한다. 임계값은각학습내부구간검증정상99%보다엄격한점수로정한다.',
'확인필터는RF기존판정에ET또는Cat점수가학습내부정상상위10%/25%경계를넘는경우만허용한다. 평가정답으로경계를고르지않음. 기존RF양성의부분집합이므로같은평가에서추가가스발견은불가능하고오탐감소/가스손실만확인한다. AND필터의AP는별도연속점수를정의하지않아산출하지않음.',
'바깥구간제외/과거평가를구분하며정상566가스17/정상427가스13. 센서이력및RPM정의불확실성조건은기존동일. 가스원인추정이아니다.',
'','| 평가 | 결합 | AP | TP | FP | FN | TN | F1 |','|---|---|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} |')
(root/'reports/rg3_small_confirm_ensemble_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(metrics.to_string(index=False))


             scheme             model       AP  TP  FP  FN  TN       F1
leave_one_block_out        RF_ET_mean 0.183384   4   8  13 558 0.275862
leave_one_block_out       RF_Cat_mean 0.178577   2   6  15 560 0.160000
leave_one_block_out    RF_ET_Cat_mean 0.177837   3   8  14 558 0.214286
leave_one_block_out    RF_half_ET_Cat 0.182588   4   7  13 559 0.285714
leave_one_block_out       RF_baseline 0.168625   4   6  13 560 0.296296
leave_one_block_out  RF_confirm_ET_10      NaN   4   6  13 560 0.296296
leave_one_block_out  RF_confirm_ET_25      NaN   4   6  13 560 0.296296
leave_one_block_out RF_confirm_Cat_10      NaN   4   6  13 560 0.296296
leave_one_block_out RF_confirm_Cat_25      NaN   4   6  13 560 0.296296
   past_blocks_only        RF_ET_mean 0.072371   2   7  11 420 0.181818
   past_blocks_only       RF_Cat_mean 0.071030   0   5  13 422 0.000000
   past_blocks_only    RF_ET_Cat_mean 0.064110   1   7  12 420 0.095238
   past_blocks_only    RF_half_ET_Cat 0.078816   2   6  11 421 0